In [ ]:
import pandas as pd
pd.options.plotting.backend = "plotly"
import h5py
import numpy as np


In [ ]:
with h5py.File('20260618-143501_fullsky.h5', 'r') as f:
    dfpos = pd.DataFrame({'ts': f['timestamp'][:], 'az': f['az'][:], 'el': f['el'][:]}).astype({'ts':np.int64})

In [ ]:
dfpos.iloc[:-100000:10][['az','el']].plot()

In [ ]:
with h5py.File('20260617-205347_xband.h5', 'r') as f:
    #spectra = f["spectra"]["spectrum"]          # shape (N, channels)
    #ts_spec = pd.to_datetime(f["spectra"]["timestamp"][:], unit="ms")
    ts = f["spectra"]["timestamp"][:]
    dfspec = pd.DataFrame({"ts": ts, "spec_idx": range(len(ts))}).astype({'ts':np.int64})

In [ ]:
merged = pd.merge_asof(
    dfspec,
    dfpos,
    on="ts",
    direction="nearest",
    tolerance=100   # adjust
)


In [ ]:
merged.ts = merged.ts // 1000
merged = merged.loc[merged.az.notna()]

In [ ]:
merged[['az','el']].plot()

In [ ]:
binned = merged.copy()
binned['el'] = (binned['el']/2+0.5).astype(int)
binned['az'] = (binned['az']/12+.5).astype(int)

In [ ]:
binned

In [ ]:
binned[['az','el']].plot()

In [ ]:
data = []

In [ ]:
with h5py.File('20260603-180131_sband.h5', 'r') as f:
    spectra = f['spectra']['spectrum']
    for (az,el), df in binned.groupby(['el','az']):
        data.append((az, el, spectra[df['spec_idx'].values].max(axis=0).max()))

In [ ]:
tmp = np.array(data[:-1])

x = tmp[:, 0].astype(int)
y = tmp[:, 1].astype(int)
v = tmp[:, 2]

arr = np.zeros((y.max() + 1, x.max() + 1))
arr[y, x] = v

In [ ]:
arr = arr[:,1:]
print(arr.shape)

In [ ]:
import plotly.express as px
fig = px.imshow(arr, labels=dict(x="elevation", y="azimuth", color="power"),
               x=[str(q) for q in np.arange(arr.shape[1])*2+1],
               y=[str(q) for q in np.arange(arr.shape[0])*12],
          aspect='equal'
            )

fig.update_yaxes(
    scaleanchor="x",
    scaleratio=.1
)

In [ ]:
import numpy as np
from scipy.interpolate import RegularGridInterpolator

# Z has shape (nr, ntheta)
interp = RegularGridInterpolator(
    (x, np.pi/180*y),
    v,
    bounds_error=False,
    fill_value=np.nan
)

N = 500
x = np.linspace(-1, 1, N)
y = np.linspace(-1, 1, N)

X, Y = np.meshgrid(x, y)

R2 = np.sqrt(X**2 + Y**2)
TH2 = np.mod(np.arctan2(Y, X), 2*np.pi)

cartesian = interp(
    np.column_stack([R2.ravel(), TH2.ravel()])
).reshape(N, N)
